# Transformer Architecture
### Interactive Notebook for AI/ML Interview Preparation

Implements attention, multi-head attention, positional encoding from scratch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
print('Libraries loaded!')

---
## 1. Scaled Dot-Product Attention

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)  # Scale
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)
    weights = np.exp(scores - scores.max(axis=-1, keepdims=True))  # softmax
    weights /= weights.sum(axis=-1, keepdims=True)
    return weights @ V, weights

# Example: 4 tokens, embedding dim = 8
seq_len, d_model = 4, 8
X = np.random.randn(seq_len, d_model)
W_Q = np.random.randn(d_model, d_model) * 0.1
W_K = np.random.randn(d_model, d_model) * 0.1
W_V = np.random.randn(d_model, d_model) * 0.1
Q, K, V = X @ W_Q, X @ W_K, X @ W_V

output, attn_weights = scaled_dot_product_attention(Q, K, V)

plt.figure(figsize=(5, 4))
plt.imshow(attn_weights, cmap='Blues')
plt.xlabel('Key position'); plt.ylabel('Query position')
plt.title('Attention Weights')
plt.colorbar(); plt.tight_layout(); plt.show()

---
## 2. Multi-Head Attention

In [ ]:
def multi_head_attention(X, n_heads, d_model):
    d_k = d_model // n_heads
    heads = []
    for _ in range(n_heads):
        W_Q = np.random.randn(d_model, d_k) * 0.1
        W_K = np.random.randn(d_model, d_k) * 0.1
        W_V = np.random.randn(d_model, d_k) * 0.1
        Q, K, V = X @ W_Q, X @ W_K, X @ W_V
        head_out, _ = scaled_dot_product_attention(Q, K, V)
        heads.append(head_out)
    concat = np.concatenate(heads, axis=-1)  # (seq_len, d_model)
    W_O = np.random.randn(d_model, d_model) * 0.1
    return concat @ W_O

mha_output = multi_head_attention(X, n_heads=4, d_model=8)
print(f'Input shape:  {X.shape}')
print(f'Output shape: {mha_output.shape} (same as input!)')

---
## 3. Positional Encoding

In [ ]:
def positional_encoding(max_len, d_model):
    PE = np.zeros((max_len, d_model))
    position = np.arange(max_len)[:, None]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    PE[:, 0::2] = np.sin(position * div_term)
    PE[:, 1::2] = np.cos(position * div_term)
    return PE

PE = positional_encoding(50, 64)
plt.figure(figsize=(10, 4))
plt.imshow(PE, aspect='auto', cmap='RdBu')
plt.xlabel('Embedding Dimension'); plt.ylabel('Position')
plt.title('Sinusoidal Positional Encoding')
plt.colorbar(); plt.tight_layout(); plt.show()

---
## 4. Causal Masking (Decoder)

In [ ]:
# Causal mask: prevent attending to future positions
seq_len = 6
causal_mask = np.tril(np.ones((seq_len, seq_len)))  # lower triangular

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(causal_mask, cmap='Blues')
axes[0].set_title('Causal Mask (1=allowed, 0=blocked)')
axes[0].set_xlabel('Key'); axes[0].set_ylabel('Query')

# Apply to attention
Q = np.random.randn(seq_len, 8)
K = np.random.randn(seq_len, 8)
V = np.random.randn(seq_len, 8)
_, masked_weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
axes[1].imshow(masked_weights, cmap='Blues')
axes[1].set_title('Masked Attention Weights')
axes[1].set_xlabel('Key'); axes[1].set_ylabel('Query')
plt.tight_layout(); plt.show()
print('Decoder: each position can only attend to itself and earlier positions')

---
## 5. Complexity Analysis

In [ ]:
seq_lengths = np.arange(100, 10001, 100)
d = 512
attention_cost = seq_lengths ** 2 * d
ffn_cost = seq_lengths * d ** 2

plt.figure(figsize=(8, 4))
plt.plot(seq_lengths, attention_cost / 1e9, label='Attention O(n²d)', color='red')
plt.plot(seq_lengths, ffn_cost / 1e9, label='FFN O(nd²)', color='blue')
plt.xlabel('Sequence Length')
plt.ylabel('Operations (billions)')
plt.title('Transformer Complexity: Attention dominates at long sequences')
plt.legend(); plt.tight_layout(); plt.show()

---
## Key Interview Takeaways

1. **Self-attention** — Attention(Q,K,V) = softmax(QKᵀ/√d)V
2. **Multi-head** — multiple attention heads capture different relationships
3. **Positional encoding** — sinusoidal functions inject position info
4. **Causal masking** — decoder can't see future tokens
5. **O(n²) complexity** — main bottleneck; drives efficient attention research

---

<small><em>© 2026 AI Nirvana · Disclaimer: Provided as is. No liability assumed.</em></small>